# 05 - IEEE-CIS Rule Explanation Evaluation

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

KAGGLE = Path("/kaggle").exists()
REPO_URL = "https://github.com/Tommyhuy1705/Explainable_NeuroSymbolic_Fraud_Detection.git"
KAGGLE_PROJECT_DIR = Path("/kaggle/working/Explainable_NeuroSymbolic_Fraud_Detection")

if KAGGLE:
    os.environ["THESIS_QUICK_RUN"] = "0"
    os.environ["THESIS_SYNTHETIC_FALLBACK"] = "0"
    if not KAGGLE_PROJECT_DIR.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(KAGGLE_PROJECT_DIR)],
            check=True,
        )

def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src").is_dir() and (candidate / "configs").is_dir():
            return candidate
    for base in (Path("/kaggle/working"), Path("/kaggle/input")):
        if base.exists():
            matches = [path for path in base.glob("**/configs") if path.is_dir()]
            if matches:
                return matches[0].parent
    raise FileNotFoundError("Project root with src/ and configs/ was not found")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

QUICK_RUN = os.getenv("THESIS_QUICK_RUN", "0") == "1"
ALLOW_SYNTHETIC_FALLBACK = os.getenv("THESIS_SYNTHETIC_FALLBACK", "0") == "1"
OUTPUT_BASE = Path("/kaggle/working/thesis_outputs") if KAGGLE else PROJECT_ROOT / "results/runs/notebooks"
INPUT_ROOTS = [OUTPUT_BASE, PROJECT_ROOT / "results/runs/notebooks"]
if Path("/kaggle/input").exists():
    INPUT_ROOTS.append(Path("/kaggle/input"))

try:
    GIT_COMMIT = subprocess.check_output(
        ["git", "-C", str(PROJECT_ROOT), "rev-parse", "HEAD"], text=True
    ).strip()
except (OSError, subprocess.CalledProcessError):
    GIT_COMMIT = None

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)
print({
    "project_root": str(PROJECT_ROOT),
    "git_commit": GIT_COMMIT,
    "quick_run": QUICK_RUN,
    "synthetic_fallback": ALLOW_SYNTHETIC_FALLBACK,
    "kaggle": KAGGLE,
})

## Thiết lập

Notebook chỉ đọc frozen IEEE-CIS predictor từ Notebook 02. Predictor probabilities,
calibration và threshold không được train/chọn lại trong bước explanation.

In [ ]:
from src.artifacts import assert_frozen_alignment, load_frozen_reference_artifact
from src.data import load_config, prepare_dataset
from src.experiment import load_experiment_data

config = load_config(PROJECT_ROOT / "configs/ieee_cis.yaml")
frame, data_source = load_experiment_data(
    config, max_rows=12000 if QUICK_RUN else None,
    synthetic_fallback=ALLOW_SYNTHETIC_FALLBACK,
    synthetic_rows=12000 if QUICK_RUN else 6000,
)
prepared = prepare_dataset(frame, config)
artifact = load_frozen_reference_artifact(
    "ieee_cis", expected_config=config,
    search_roots=[OUTPUT_BASE / "02_ieee_cis_model_benchmarks", *INPUT_ROOTS],
)
assert_frozen_alignment(artifact, prepared.y_validation, prepared.y_test)
if bool(artifact["manifest"]["quick_run"]) != QUICK_RUN:
    raise ValueError("Notebook mode and frozen artifact quick_run flag do not match")
probabilities = artifact["test_probability"]
threshold = float(artifact["manifest"]["threshold"])
print({
    "data_source": data_source,
    "reference_model": artifact["manifest"]["model"],
    "reference_seed": artifact["manifest"]["reference_seed"],
    "calibration_method": artifact["manifest"]["calibration_method"],
    "threshold": threshold,
    "artifact": str(artifact["artifact_path"]),
})

In [ ]:
from src.explanation import (
    RuleExplainer, bootstrap_explanation_precision_gain,
    explanation_quality_metrics, rule_quality_table,
)
from src.logic import FraudRuleEngine

output_dir = OUTPUT_BASE / "05_ieee_cis_rule_explanation_evaluation"
output_dir.mkdir(parents=True, exist_ok=True)
target = config["dataset"]["target_column"]
engine = FraudRuleEngine(config["logic"]["rules"]).fit(prepared.train_frame, target)
explainer = RuleExplainer(engine, config["logic"]["activation_threshold"], config["logic"]["top_k_rules"])
explanations = explainer.explain(prepared.test_frame, probabilities, threshold)
explanations["y_true"] = prepared.y_test
display(explanations.head())

## Results

In [ ]:
truth = engine.evaluate(prepared.test_frame)
rule_quality = rule_quality_table(truth, prepared.y_test, config["logic"]["activation_threshold"])
quality = explanation_quality_metrics(explanations, prepared.y_test, probabilities, threshold)
quality.update(bootstrap_explanation_precision_gain(
    explanations, prepared.y_test, probabilities, threshold,
    n_bootstrap=config["evaluation"]["bootstrap_iterations"], seed=config["project"]["seed"],
))
quality_frame = pd.DataFrame([quality])
display(rule_quality.round(4), quality_frame.round(4))

predicted_alert = probabilities >= threshold
cases = explanations.copy()
cases["case_type"] = np.select(
    [predicted_alert & (prepared.y_test == 1), predicted_alert & (prepared.y_test == 0),
     (~predicted_alert) & (prepared.y_test == 1)],
    ["true_positive", "false_positive", "false_negative"], default="true_negative",
)
selected_cases = pd.concat([
    group.sort_values("predicted_probability", ascending=False).head(3)
    for _, group in cases.groupby("case_type")
])
display(selected_cases[["case_type", "y_true", "predicted_probability", "rule_names", "explanation"]])
quality_frame.to_csv(output_dir / "ieee_explanation_quality.csv", index=False)
rule_quality.to_csv(output_dir / "ieee_test_rule_quality.csv", index=False)
selected_cases.to_json(output_dir / "ieee_explanation_cases.json", orient="records", indent=2)

## Takeaways

In [ ]:
display(Markdown(
    f"- Alert explanation coverage: **{quality['explanation_coverage_alerts']:.3f}**.\n"
    f"- Explained-alert precision gain: **{quality['explained_alert_precision_gain']:.3f}** "
    f"(95% CI [{quality['precision_gain_ci_low']:.3f}, {quality['precision_gain_ci_high']:.3f}]).\n"
    f"- Contradiction rate: **{quality['contradiction_rate']:.3f}**.\n"
    "- These are rule-evidence diagnostics, not causal explanations."
))